# Capstone Track B — Fine-Tuned Model Showcase
**Day 2 Afternoon | ~3 hours | Colab T4 GPU required ⚡**

---

## ⚠️ Switch to T4 GPU First
```
Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save
Then re-run from the top.
```

## What You Are Building
A before/after comparison of a base model vs a QLoRA fine-tuned version on a **narrow task you choose**. You will present both models side-by-side in a Gradio interface.

**Your job in this template:**
- [ ] Choose a narrow, demonstrable task (Step 1)
- [ ] Write 15–25 training examples in the required format (Step 1)
- [ ] Choose LoRA rank and justify it (Step 3)
- [ ] Write 5 test prompts that show clear before/after contrast (Step 5)

Everything else (model loading, NF4 config, SFTTrainer, Gradio shell) is provided.

> **Track A (RAG)?** Open `capstone_track_a.ipynb` instead — it runs on free CPU.

In [ ]:
%%capture
!pip install transformers torch accelerate bitsandbytes peft trl datasets gradio
print('Done')

In [ ]:
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✅ GPU detected:', result.stdout.strip())
else:
    print('❌ No GPU found. Go to Runtime → Change runtime type → T4 GPU → Save, then re-run.')

In [ ]:
# ─── CONFIGURATION ───────────────────────────────────────────────────────────
MODEL_ID    = 'Qwen/Qwen2.5-0.5B-Instruct'   # reliable on T4; change to 1.5B if VRAM allows
OUTPUT_DIR  = './capstone_lora'
ADAPTER_DIR = './capstone_adapter'
# ─────────────────────────────────────────────────────────────────────────────
print('✅ Config loaded')

---
## Step 1 — Choose Your Task and Write Training Examples

**Rule: narrow tasks show clear before/after contrast.** 

Good task choices:
- Always respond in a specific JSON schema
- Translate technical jargon into plain language for a specific domain
- Classify text into your own custom categories
- Formal letter / email writing with a specific structure
- A domain-specific Q&A style (always start with 'According to...')

Bad task choices:
- 'Be smarter / more helpful' — too vague, no measurable before/after
- General knowledge tasks — the base model already does these well

**You need 15–25 examples minimum.** Quality matters more than quantity. All examples should follow the exact same input/output pattern.

In [ ]:
# ── TODO: Define your task and write training examples ──────────────────────
# Each example: {'instruction': '<your prompt>', 'response': '<ideal output>'}
# The instruction is what you'll send to the model.
# The response is what you want it to learn to produce.

TASK_DESCRIPTION = 'DESCRIBE YOUR TASK HERE'
# Example: 'Given a technical error message, explain it in plain English for a non-developer.'

training_examples = [
    # TODO: replace ALL of these with real examples from your task
    # Add at least 15 examples
    {'instruction': 'EXAMPLE INPUT 1',  'response': 'IDEAL OUTPUT 1'},
    {'instruction': 'EXAMPLE INPUT 2',  'response': 'IDEAL OUTPUT 2'},
    {'instruction': 'EXAMPLE INPUT 3',  'response': 'IDEAL OUTPUT 3'},
    {'instruction': 'EXAMPLE INPUT 4',  'response': 'IDEAL OUTPUT 4'},
    {'instruction': 'EXAMPLE INPUT 5',  'response': 'IDEAL OUTPUT 5'},
    # ... keep going to 15-25 total
]

# Hold out the last 3 for testing — don't train on these
train_data = training_examples[:-3]
test_data  = training_examples[-3:]

print(f'Task: {TASK_DESCRIPTION}')
print(f'Training examples: {len(train_data)}')
print(f'Test examples    : {len(test_data)}')

---
## Step 2 — Load & Quantize the Base Model

Loading the base model in NF4 (4-bit) so it fits on a T4 GPU. This is identical to what you did in Lab 4 — provided here as-is.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading {MODEL_ID} in NF4...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto')
print(f'✅ Model loaded')
print(f'   Parameters : {sum(p.numel() for p in base_model.parameters()) / 1e6:.0f}M')
if torch.cuda.is_available():
    print(f'   VRAM used  : {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
# Test the BASE model before fine-tuning — save this output for comparison
def chat_base(prompt, max_new_tokens=200):
    msgs = [{'role': 'user', 'content': prompt}]
    ids  = tokenizer.apply_chat_template(msgs, return_tensors='pt',
                                         add_generation_prompt=True).to(base_model.device)
    with torch.no_grad():
        out = base_model.generate(ids, max_new_tokens=max_new_tokens,
                                  temperature=0.1, do_sample=True)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

# TODO: replace with a prompt from your task
test_prompt = 'YOUR TEST PROMPT HERE'
print('Base model response:')
print(chat_base(test_prompt))

---
## Step 3 — Attach LoRA Adapters

LoRA freezes the base model and trains two small matrices (A and B) per layer. The rank `r` controls how many parameters are trainable.

| Rank | Trainable params (0.5B model) | When to use |
|------|-------------------------------|-------------|
| 4    | ~0.3% of total                | Very narrow tasks, few examples |
| 8    | ~0.7% of total                | Most tasks — good default |
| 16   | ~1.4% of total                | Complex tasks, 50+ examples |

**TODO:** Choose your rank below and explain why in the comment.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)

# TODO: choose rank and explain why in this comment
# My task is [X], so I chose rank [Y] because [Z]
LORA_RANK   = 8    # TODO: adjust based on task complexity
LORA_ALPHA  = 16   # scaling factor — typically 2x rank
LORA_TARGET = ['q_proj', 'v_proj']  # which weight matrices to adapt

lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET,
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
)
model_peft = get_peft_model(base_model, lora_config)

total  = sum(p.numel() for p in model_peft.parameters())
trained = sum(p.numel() for p in model_peft.parameters() if p.requires_grad)
print(f'Trainable parameters: {trained:,} / {total:,} ({100*trained/total:.2f}%)')

---
## Step 4 — Fine-Tune

The SFTTrainer config is set up for a T4 with 15 GB VRAM. 3 epochs on 15–25 examples takes about 3–8 minutes.

In [ ]:
from datasets import Dataset

# Format training examples into the chat template format
def format_example(ex):
    msgs = [
        {'role': 'user',      'content': ex['instruction']},
        {'role': 'assistant', 'content': ex['response']},
    ]
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False)}

train_dataset = Dataset.from_list(train_data).map(format_example)
print(f'Training dataset: {len(train_dataset)} examples')
print('Sample formatted:')
print(train_dataset[0]['text'][:300], '...')

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_strategy='no',
    report_to='none',
    dataset_text_field='text',
    max_seq_length=512,
)

trainer = SFTTrainer(
    model=model_peft,
    args=training_args,
    train_dataset=train_dataset,
)

print('Starting training...')
trainer.train()
model_peft.save_pretrained(ADAPTER_DIR)
print(f'✅ Training complete. Adapter saved to {ADAPTER_DIR}')

---
## Step 5 — Compare Base vs Fine-Tuned

Load the fine-tuned adapter alongside the base model and run your held-out test prompts. A strong capstone shows **clear, specific** before/after contrast — not subtle differences.

If the difference is too subtle, consider:
- Adding more training examples
- Increasing the rank (`LORA_RANK`)
- Narrowing the task further
- Increasing epochs to 5

In [ ]:
from peft import PeftModel

# Reload base model (fresh, unmodified)
print('Reloading base model...')
base_reload = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto')
finetuned_model = PeftModel.from_pretrained(base_reload, ADAPTER_DIR)
print('✅ Fine-tuned model ready')

def chat_finetuned(prompt, max_new_tokens=200):
    msgs = [{'role': 'user', 'content': prompt}]
    ids  = tokenizer.apply_chat_template(msgs, return_tensors='pt',
                                         add_generation_prompt=True).to(finetuned_model.device)
    with torch.no_grad():
        out = finetuned_model.generate(ids, max_new_tokens=max_new_tokens,
                                       temperature=0.1, do_sample=True)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

# Run comparison on held-out test examples
print('=' * 70)
for i, ex in enumerate(test_data, 1):
    prompt = ex['instruction']
    print(f'\n[Test {i}] {prompt}')
    print(f'Expected  : {ex["response"][:100]}')
    print(f'Base      : {chat_base(prompt)[:100]}')
    print(f'Fine-tuned: {chat_finetuned(prompt)[:100]}')
    print('-' * 70)

---
## Step 6 — Build the Comparison App

The Gradio UI below shows both models side by side. **Your job:** update `EXAMPLE_PROMPTS` with real prompts from your task.

In [ ]:
import gradio as gr

# TODO: replace with real prompts from your task
EXAMPLE_PROMPTS = [
    'YOUR TEST PROMPT 1',
    'YOUR TEST PROMPT 2',
    'YOUR TEST PROMPT 3',
]

def compare(prompt):
    base_out   = chat_base(prompt)
    tuned_out  = chat_finetuned(prompt)
    return base_out, tuned_out

with gr.Blocks(title=f'Before/After: {TASK_DESCRIPTION}') as demo:
    gr.Markdown(f'# Fine-Tuning Showcase\n**Task:** {TASK_DESCRIPTION}')
    prompt_box = gr.Textbox(label='Test prompt', lines=3)
    with gr.Row():
        base_box  = gr.Textbox(label=f'Base: {MODEL_ID}', lines=10, interactive=False)
        tuned_box = gr.Textbox(label='Fine-Tuned (QLoRA)',  lines=10, interactive=False)
    gr.Button('Compare').click(compare, [prompt_box], [base_box, tuned_box])
    gr.Examples(EXAMPLE_PROMPTS, inputs=prompt_box)

demo.launch(share=True)

---
## Submission Checklist

Before your presentation, verify:

- [ ] My task is narrow and shows clear before/after contrast
- [ ] I have at least 15 training examples
- [ ] The fine-tuned model behaves noticeably differently on test prompts
- [ ] The Gradio comparison app is live with a public URL
- [ ] I can answer: what rank did I use and why?
- [ ] I can describe one thing that didn't work as expected

**Presentation structure (5 minutes):**
1. *'I fine-tuned [model] to [task] because [reason].'* (30 sec)
2. Architecture: dataset format → NF4 base → LoRA adapters → SFTTrainer → adapter save (60 sec)
3. Live demo in Gradio: 2 prompts that show clear difference + 1 that doesn't (2 min)
4. What surprised you / what you'd change (90 sec)